In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore", message="Workbook contains no default style")

# ============================================================
# 1. LOAD BIOLOGY DATA
# ============================================================
df_bio = pd.read_csv("Datasets/df_bio_2022_2025.csv")

# ============================================================
# 2. CLEANING AND BINARY VARIABLE GENERATION
# ============================================================

# --- Blood Gas (GDS) ---
cols_gds_values = [c for c in df_bio.columns if c.startswith("gds_") and "origin" not in c]
mask_origin_present = df_bio["gds_origin"].notna()
mask_no_values = df_bio[cols_gds_values].isna().all(axis=1)

# Handle "Ghost" patients (origin exists but no values)
df_bio.loc[mask_origin_present & mask_no_values, "gds_origin"] = np.nan

cols_gds = [c for c in df_bio.columns if c.startswith("gds_") or c.startswith("labo_gds_")]
df_bio["has_blood_gas"] = df_bio[cols_gds].notna().any(axis=1).astype(int)

# --- Anticoagulants (aXa / aIIa) ---
cols_anti = ["axa_hnf", "axa_arixtra", "axa_eliquis", "axa_hbpm", "axa_xarelto", "aiia_pradaxa"]
df_bio["is_aXa_aIIa"] = df_bio[cols_anti].notna().any(axis=1).astype(int)

# --- Cerebrospinal Fluid (CSF / LP) ---
cols_lcr = [c for c in df_bio.columns if "lcr_aspect" in c]
df_bio["has_lumbar_puncture"] = df_bio[cols_lcr].notna().any(axis=1).astype(int)

# --- Lactates ---
cols_lactate = [c for c in df_bio.columns if "lactate" in c]
df_bio["is_lactates"] = df_bio[cols_lactate].notna().any(axis=1).astype(int)

# --- Cultures ---
df_bio["has_culture"] = df_bio["culture"].notna().astype(int)

# --- Leucocytes & Differential ---
df_bio["is_leucocytes"] = df_bio["leucocytes"].notna().astype(int)
cols_diff = ["neutrophils", "lymphocytes", "monocytes", "eosinophils", "basophils"]
df_bio["is_formule_leuco"] = df_bio[cols_diff].notna().any(axis=1).astype(int)

# --- Other Standard Lab Variables ---
bio_vars = [
    "alat", "asat", "bnp", "bili_total", "ck", "ckmb", "crp",
    "calcium_ionized", "calcium", "creatinine", "ddimer", "iron",
    "ferritin", "fibrinogen", "hemoglobine", "lipase", "alp",
    "platelets", "potassium", "pct", "sodium", "aptt", "pt",
    "troponine", "urea"
]

for var in bio_vars:
    if var in df_bio.columns:
        df_bio[f"is_{var}"] = df_bio[var].notna().astype(int)

# ============================================================
# 3. AGGREGATED FEATURES & COMPLEXITY SCORE
# ============================================================

# Define all columns that belong to a standard veinous blood test
cols_veinous_analysis = [c for c in df_bio.columns if c.startswith("is_")]

# 1. Flag for at least one blood test performed
df_bio['has_blood_test'] = df_bio[cols_veinous_analysis].any(axis=1).astype(int)

# 2. Biological Complexity Score (0 to 4)
# Defining the 4 pillars of bio investigation
bio_pillars = ['has_blood_test', 'has_lumbar_puncture', 'has_culture', 'has_blood_gas']
df_bio['bio_exam_count'] = df_bio[bio_pillars].sum(axis=1)

# ============================================================
# 4. SAVE CLEANED BIOLOGY DATA
# ============================================================
bio_clean_file = "Datasets/df_bio_clean_with_binaries_pel22.csv"
df_bio.to_csv(bio_clean_file, index=False, sep=';', encoding='utf-8')
print(f"✅ Biology file saved: {bio_clean_file}")

# ============================================================
# 5. MERGE WITH MASTER DATASET
# ============================================================
print("\n🔗 Starting Merge Process...")

df_master = pd.read_csv("Datasets/df_adm_pv_ioa_med_radio_tabular.csv", dtype={'nda': str})
df_master['nda'] = df_master['nda'].astype(str).str.strip()
df_bio['nda'] = df_bio['nda'].astype(str).str.strip()

# Create a copy for merging and add a lab presence marker
df_bio_merge = df_bio.copy()
df_bio_merge['is_labo'] = 1

# Left Join: Keep all patients from the Master Dataset
df_final = pd.merge(
    df_master,
    df_bio_merge,
    on='nda',
    how='left',
    suffixes=('', '_bio')
)

# ============================================================
# 6. POST-MERGE HARMONIZATION
# ============================================================

# Fill all binary (is_/has_) and count variables with 0 for patients without lab data
binary_and_score_cols = [c for c in df_final.columns if c.startswith(('is_', 'has_', 'bio_'))]
df_final[binary_and_score_cols] = df_final[binary_and_score_cols].fillna(0).astype(int)

# ============================================================
# 7. FINAL REPORT & SAVE
# ============================================================
print("\n" + "="*50)
print("📊 FINAL MERGE SUMMARY")
print("="*50)
print(f"Master dataset size       : {len(df_master)}")
print(f"Patients with Bio merged  : {df_final['is_labo'].sum()}")
print(f"Final columns count       : {len(df_final.columns)}")
print("="*50)

output_file = "Datasets/df_adm_pv_ioa_med_radio_labo_tabular.csv"
df_final.to_csv(output_file, index=False)
print(f"\n✅ Final consolidated dataset saved as: {output_file}")

/tmp/ipykernel_626892/577302742.py:10: DtypeWarning:

Columns (0,4,6,7,9,10,11,21,22,26,37) have mixed types. Specify dtype option on import or set low_memory=False.



✅ Biology file saved: Datasets/df_bio_clean_with_binaries_pel22.csv

🔗 Starting Merge Process...


/tmp/ipykernel_626892/577302742.py:87: DtypeWarning:

Columns (12,78,81,84,85,87,88,89,90,91,92) have mixed types. Specify dtype option on import or set low_memory=False.




📊 FINAL MERGE SUMMARY
Master dataset size       : 123189
Patients with Bio merged  : 73963
Final columns count       : 181



✅ Final consolidated dataset saved as: Datasets/df_adm_pv_ioa_med_radio_labo_tabular.csv


In [2]:
nda_bio    = set(df_bio['nda'].astype(str).str.strip().unique())
nda_master = set(df_master['nda'].astype(str).str.strip().unique())

bio_not_in_master = nda_bio - nda_master

print(f"Patients in df_bio                  : {len(nda_bio):,}")
print(f"Patients in master                  : {len(nda_master):,}")
print(f"Patients in df_bio NOT in master    : {len(bio_not_in_master):,}")
print(f"Patients in df_bio AND in master    : {len(nda_bio & nda_master):,}")


Patients in df_bio                  : 95,582
Patients in master                  : 123,189
Patients in df_bio NOT in master    : 21,619
Patients in df_bio AND in master    : 73,963


In [3]:
# import pandas as pd
# import numpy as np
# import warnings
#
# warnings.filterwarnings("ignore", message="Workbook contains no default style")
#
# # ============================================================
# # 1. LOAD BIOLOGY DATA
# # ============================================================
# df_bio = pd.read_csv("df_bio_subset22pelglims.csv")
#
#
#
# # ============================================================
# # 3. GDS (Blood Gas) — CLEANING AND BINARY VARIABLE
# # ============================================================
# cols_gds_values = [c for c in df_bio.columns if c.startswith("gds_") and "origin" not in c]
#
# mask_origin_present = df_bio["gds_origin"].notna()
# mask_no_values = df_bio[cols_gds_values].isna().all(axis=1)
#
# ghost_patients = df_bio[mask_origin_present & mask_no_values]
#
# print("📊 BLOOD GAS ORIGIN DIAGNOSTIC")
# print(f"Patients with origin: {mask_origin_present.sum()}")
# print(f"Ghost patients (origin but no values): {len(ghost_patients)}")
#
# df_bio.loc[ghost_patients.index, "gds_origin"] = np.nan
#
# cols_gds = [c for c in df_bio.columns if c.startswith("gds_") or c.startswith("labo_gds_")]
# df_bio[("is_blood_gas")] = df_bio[cols_gds].notna().any(axis=1).astype(int)
#
# # ============================================================
# # 4. ANTICOAGULANTS (aXa / aIIa)
# # ============================================================
# cols_anti = ["axa_hnf", "axa_arixtra", "axa_eliquis", "axa_hbpm", "axa_xarelto", "aiia_pradaxa"]
# df_bio["is_aXa_aIIa"] = df_bio[cols_anti].notna().any(axis=1).astype(int)
#
# # ============================================================
# # 5. LCR (CSF)
# # ============================================================
# cols_lcr = [c for c in df_bio.columns if "lcr_aspect" in c]
# df_bio["is_csf"] = df_bio[cols_lcr].notna().any(axis=1).astype(int)
#
# # ============================================================
# # 6. LACTATE (Arterial or Venous)
# # ============================================================
# cols_lactate = [c for c in df_bio.columns if "lactate" in c]
# df_bio["is_lactates"] = df_bio[cols_lactate].notna().any(axis=1).astype(int)
#
# # ============================================================
# # 7. CULTURES
# # ============================================================
# df_bio["is_culture"] = df_bio["culture"].notna().astype(int)
#
# # ============================================================
# # 8. LEUCOCYTES + DIFFERENTIAL
# # ============================================================
# df_bio["is_leucocytes"] = df_bio["leucocytes"].notna().astype(int)
#
# cols_diff = ["neutrophils", "lymphocytes", "monocytes", "eosinophils", "basophils"]
# df_bio["is_formule_leuco"] = df_bio[cols_diff].notna().any(axis=1).astype(int)
#
# # ============================================================
# # 9. OTHER BINARY VARIABLES (is_*)
# # ============================================================
# bio_vars = [
#     "alat", "asat", "bnp", "bili_total", "ck", "ckmb", "crp",
#     "calcium_ionized", "calcium", "creatinine", "ddimer", "iron",
#     "ferritin", "fibrinogen", "hemoglobine", "lipase", "alp",
#     "platelets", "potassium", "pct", "sodium", "aptt", "pt",
#     "troponine", "urea"
# ]
#
# for var in bio_vars:
#     if var in df_bio.columns:
#         df_bio[f"is_{var}"] = df_bio[var].notna().astype(int)
#     else:
#         print(f"⚠️ Missing column: {var}")
#
# # ============================================================
# # 10. SUMMARY OF ALL is_* VARIABLES
# # ============================================================
# all_is_cols = [c for c in df_bio.columns if c.startswith("is_")]
#
# summary = df_bio[all_is_cols].agg(["sum", "mean"]).T
# summary.columns = ["Count", "Frequency (%)"]
# summary["Frequency (%)"] = (summary["Frequency (%)"] * 100).round(2)
#
# print("\n📊 SUMMARY OF BINARY VARIABLES (is_*)")
# print(summary.sort_values(by="Count", ascending=False))
#
# print(f"\nTotal number of binary variables detected: {len(all_is_cols)}")


In [4]:
# # SAUVEGARDE DU DATAFRAME NETTOYÉ ET AUGMENTÉ
#
# # On définit le nom du fichier avec la date ou un tag de version
# file_name = "df_bio_clean_with_binaries_pel22.csv"
#
# # Export en CSV
# # index=False est important pour ne pas créer une colonne "Unnamed: 0" à la prochaine lecture
# df_bio.to_csv(file_name, index=False, sep=';', encoding='utf-8')
#
# print(f"✅ Sauvegarde réussie : {file_name}")
# print(f"Dimensions du fichier : {df_bio.shape[0]} lignes x {df_bio.shape[1]} colonnes")

In [5]:
# # ========================================
# # MERGE WITH MASTER CSV (IOA + PV + MED + RADIO)
# # ========================================
#
# import pandas as pd
# import numpy as np
#
# # ============================================================
# # 1. LOAD MAIN DATASET AND BIOLOGY DATA
# # ============================================================
# df_principal = pd.read_csv("df_adm_pv_ioa_med_radio_pel22_tabular.csv", dtype={'nda': str})
# # 1. On force le séparateur (souvent ';' en France) et on gère les lignes corrompues
# df_bio = pd.read_csv(
#     "df_bio_clean_with_binaries_pel22.csv",
#     sep=None,             # Détection automatique du séparateur (virgule ou point-virgule)
#     engine='python',      # Plus lent mais gère mieux les erreurs de parsing que le moteur C
#     on_bad_lines='warn',  # Saute les lignes problématiques et t'affiche un avertissement
#     dtype={'nda': str}
# )
#
# # ============================================================
# # 2. CLEAN NDA IDENTIFIERS
# # ============================================================
# df_principal['nda'] = df_principal['nda'].astype(str).str.strip()
# df_bio['nda'] = df_bio['nda'].astype(str).str.strip()
#
# # ============================================================
# # 3. PREPARE BIOLOGY SUBSET FOR MERGE
# # ============================================================
# # Instead of filtering for 'is_', we take the whole dataframe
# df_bio_to_merge = df_bio.copy()
#
# # We add the marker to know who had a blood test
# df_bio_to_merge['is_labo'] = 1
#
# # ============================================================
# # 4. IDENTIFY BIO PATIENTS NOT PRESENT IN MAIN DATASET
# # ============================================================
# nda_bio_only = set(df_bio['nda']) - set(df_principal['nda'])
# nb_bio_only = len(nda_bio_only)
#
# # ============================================================
# # 5. MERGE (LEFT JOIN: MAIN DATASET IS THE REFERENCE)
# # ============================================================
# df_final = pd.merge(
#     df_principal,
#     df_bio_to_merge,
#     on='nda',
#     how='left',
#     suffixes=('', '_bio')
# )
#
# # ============================================================
# # 6. FILL MISSING BINARY VARIABLES WITH 0
# # ============================================================
# all_is_cols = list(set([c for c in df_final.columns if c.startswith('is_')]))
# df_final[all_is_cols] = df_final[all_is_cols].fillna(0).astype(int)
#
# # ============================================================
# # 7. SUMMARY REPORT
# # ============================================================
# print("\n" + "="*50)
# print("📊 MERGE SUMMARY (BIOLOGY + MASTER DATASET)")
# print("="*50)
# print(f"Total patients in main dataset     : {len(df_principal)}")
# print(f"Total patients in biology dataset  : {len(df_bio)}")
# print(f"Patients with biology data merged  : {df_final['is_labo'].sum()}")
# print(f"Biology-only patients (excluded)   : {nb_bio_only}")
# print(f"Final dataset size                 : {len(df_final)}")
# print("="*50)
#
# # ============================================================
# # 8. INSPECT FINAL STRUCTURE
# # ============================================================
# print("\n--- ALL COLUMNS ---")
# print(df_final.columns.tolist())
#
# cols_is = [c for c in df_final.columns if c.startswith('is_')]
# print(f"\n--- BINARY VARIABLES ({len(cols_is)}) ---")
# print(cols_is)
#
# print(f"\nFinal dimensions: {df_final.shape[0]} rows × {df_final.shape[1]} columns")
#
# # ============================================================
# # 9. SAVE FINAL MERGED DATASET
# # ============================================================
# df_final.to_csv("df_adm_pv_ioa_med_radio_labo_pel22_tabular.csv", index=False)
# print("\n✅ Final dataset saved as 'df_adm_pv_ioa_med_radio_labo_pel22_tabular.csv'.")


evidemment j'ai 12000 patient qui ont des bio, mais aui n'ont pas de dossier admin ou med ou ioa.....